# 07 — Débogage

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- dépister un bug avec des `print` bien placés (et leurs limites) ;
- utiliser `breakpoint()` pour entrer dans le débogueur intégré ;
- connaître les commandes de base de `pdb` ;
- utiliser le `f"{var=}"` pour un print de debug rapide.

## Prérequis

- toutes les notions précédentes ;
- exceptions.

Pas encore vus :

- modules standard (`logging` est en J5).

## Plan

1. Méthode : isoler, reproduire, formuler
2. Print-debug
3. `f"{var=}"` pour un debug rapide
4. `breakpoint()`
5. Commandes pdb de base
6. Débogage dans un notebook Jupyter
7. Quand passer à `logging`
8. Synthèse
9. Exercices

---


## 1. Méthode : isoler, reproduire, formuler

Avant tout outil, trois étapes mentales :

1. **Isoler** : quel est le plus petit morceau de code qui déclenche le bug ?
2. **Reproduire** : le faire apparaître *à la demande*, pas par hasard.
3. **Formuler** : décrire l'écart entre *ce qu'on attend* et *ce qu'on observe*.

Un bug bien formulé est à moitié résolu.

---


## 2. Print-debug

La méthode la plus rapide : ajouter des `print` aux endroits clés, exécuter, comparer, retirer.

In [ ]:
def fact(n: int) -> int:
    resultat = 1
    for i in range(1, n + 1):
        print('étape', i, 'résultat intermédiaire', resultat)
        resultat = resultat * i
    return resultat

print('final :', fact(5))

### Limites du print-debug

- on oublie des `print` qui traînent dans le code de prod ;
- on ne peut pas inspecter librement l'état *au moment* du bug ;
- on doit relancer le programme à chaque hypothèse.

Pour un bug simple, c'est parfait. Pour un bug coriace, passer au débogueur.

---


## 3. `f"{var=}"` — astuce Python 3.8+

L'écriture `f"{var=}"` affiche à la fois le **nom** et la **valeur** de la variable. C'est la forme idéale pour un print de debug.

In [ ]:
x = 42
nom = 'Alice'
print(f'{x=}')
print(f'{nom=}')
print(f'{x + 1=}')

---


## 4. `breakpoint()` — le point d'arrêt universel

Depuis Python 3.7, `breakpoint()` est **la** manière recommandée d'insérer un point d'arrêt. Il invoque `pdb` par défaut (ou un autre débogueur selon `PYTHONBREAKPOINT`).

### Exemple (ne pas exécuter dans un notebook automatique)

```python
def fact(n: int) -> int:
    resultat = 1
    for i in range(1, n + 1):
        breakpoint()   # on s'arrête ici
        resultat = resultat * i
    return resultat
```

À l'arrêt, Python ouvre un prompt `(Pdb) `. Vous pouvez inspecter `i`, `resultat`, modifier des variables, continuer pas à pas.

---


## 5. Commandes pdb de base

| Commande | Abréviation | Effet |
|---|---|---|
| `help` | `h` | Liste des commandes |
| `list` | `l` | Affiche le code autour du point courant |
| `next` | `n` | Pas **suivant** (sans entrer dans les appels) |
| `step` | `s` | Pas **dans** (entre dans les fonctions) |
| `continue` | `c` | Reprend jusqu'au prochain `breakpoint` |
| `print x` | `p x` | Affiche la valeur de `x` |
| `pp x` |  | `pretty print` |
| `where` | `w` | Affiche la pile d'appels |
| `up` / `down` | `u` / `d` | Monte / descend dans la pile |
| `quit` | `q` | Sort du débogueur |

### Depuis Python 3.14

`pdb` a été enrichi (historique, coloration). La table ci-dessus reste valable.

---


## 6. Débogage dans un notebook Jupyter

Jupyter offre la magie `%debug` qui lance `pdb` **post-mortem** sur la dernière exception.

In [ ]:
def diviser(a: float, b: float) -> float:
    return a / b

# Décommenter pour tester :
# diviser(10, 0)

Après l'exception, exécuter `%debug` dans la cellule suivante. On arrive dans le cadre où l'erreur a eu lieu, avec accès à toutes les variables.

### `%pdb on` — passage automatique au débogueur

`%pdb on` demande à Jupyter d'entrer dans `pdb` dès qu'une exception survient.

---


## 7. Quand passer à `logging`

`print` et `breakpoint` sont des outils de **dev**. Dans un programme qui tourne en production, on utilise `logging` — plus contrôlable (niveaux, fichiers, format). Nous le voyons au chapitre bonnes pratiques (J5).

---


## 8. Synthèse

| Outil | Quand |
|---|---|
| `print(f'{var=}')` | Petit bug rapide |
| `breakpoint()` | Inspection interactive |
| `%debug` (Jupyter) | Post-mortem d'une exception |
| `logging` | Programme qui tourne en prod |

### Règles

1. Retirer les `print` de debug avant de commiter.
2. `breakpoint()` plutôt que `import pdb; pdb.set_trace()`.
3. Un bug est toujours plus rapide à résoudre avec un test qui le reproduit.

---


## 9. Exercices

### Exercice 1 — Trouver l'erreur 1 *(facile)*

La fonction suivante est censée renvoyer la moyenne, mais elle est buggée :

```python
def moyenne(valeurs: list[float]) -> float:
    total = 0.0
    for v in valeurs:
        total = v
    return total / len(valeurs)
```

Ajouter un `print(f'{total=}')` au bon endroit, diagnostiquer, puis corriger.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def moyenne(valeurs: list[float]) -> float:
    """Moyenne d'une liste non vide."""
    total = 0.0
    for v in valeurs:
        total = total + v
    return total / len(valeurs)

print(moyenne([1.0, 2.0, 3.0]))
```

</details>

### Exercice 2 — Trouver l'erreur 2 *(facile)*

```python
def plus_grand(valeurs: list[int]) -> int:
    maxi = 0
    for v in valeurs:
        if v > maxi:
            maxi = v
    return maxi
```

Sur `[-5, -2, -9]`, la fonction renvoie `0`, ce qui est faux. Corriger.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def plus_grand(valeurs: list[int]) -> int:
    """Renvoie le maximum d'une liste non vide."""
    if len(valeurs) == 0:
        raise ValueError('liste vide')
    maxi = valeurs[0]
    for v in valeurs[1:]:
        if v > maxi:
            maxi = v
    return maxi

print(plus_grand([-5, -2, -9]))
```

</details>

### Exercice 3 — Utiliser `f"{var=}"` *(facile)*

Écrire une fonction `pythagore(a: float, b: float) -> float` qui renvoie `sqrt(a² + b²)`. Y placer un print de debug `f'{a=} {b=}'`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import math

def pythagore(a: float, b: float) -> float:
    print(f'{a=} {b=}')
    return math.sqrt(a * a + b * b)

print(pythagore(3.0, 4.0))
```

</details>

### Exercice 4 — Refactor sans print *(moyen)*

Prendre la fonction `moyenne` corrigée de l'exercice 1 et s'assurer qu'aucun `print` de debug ne subsiste dans la version finale.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
def moyenne(valeurs: list[float]) -> float:
    """Moyenne d'une liste non vide."""
    if len(valeurs) == 0:
        raise ValueError('liste vide')
    return sum(valeurs) / len(valeurs)

print(moyenne([1.0, 2.0, 3.0]))
```

</details>

### Exercice 5 — Reproduire un bug *(moyen)*

La fonction suivante semble marcher la première fois et renvoyer un résultat faux au second appel :

```python
def ajouter_tag(element: str, tags: list[str] = []) -> list[str]:
    tags.append(element)
    return tags
```

Appeler la fonction deux fois sans second argument et constater. Puis corriger en utilisant `None`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
def ajouter_tag(element: str, tags: list[str] | None = None) -> list[str]:
    """Ajoute `element` à `tags` (crée une nouvelle liste si None)."""
    if tags is None:
        tags = []
    tags.append(element)
    return tags

print(ajouter_tag('a'))
print(ajouter_tag('b'))
```

</details>

### Exercice 6 — Dichotomie buggée *(difficile)*

La recherche dichotomique suivante fonctionne parfois mais pas toujours. Ajouter un `breakpoint()` (ou des `print(f'{lo=} {hi=} {mid=}')`) et identifier le bug.

```python
def dicho(valeurs: list[int], cible: int) -> int:
    lo = 0
    hi = len(valeurs)
    while lo < hi:
        mid = (lo + hi) // 2
        if valeurs[mid] == cible:
            return mid
        if valeurs[mid] < cible:
            lo = mid
        else:
            hi = mid
    return -1
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Debugging", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
def dicho(valeurs: list[int], cible: int) -> int:
    """Recherche dichotomique, renvoie l'indice ou -1."""
    lo = 0
    hi = len(valeurs)
    while lo < hi:
        mid = (lo + hi) // 2
        if valeurs[mid] == cible:
            return mid
        if valeurs[mid] < cible:
            lo = mid + 1  # le bug était ici : il faut avancer de 1
        else:
            hi = mid
    return -1

print(dicho([1, 3, 5, 7, 9, 11], 7))
print(dicho([1, 3, 5, 7, 9, 11], 4))
```

</details>

---


## Ressources externes

- [`pdb` — doc Python](https://docs.python.org/3/library/pdb.html)
- [`breakpoint()` — PEP 553](https://peps.python.org/pep-0553/)
- [Jupyter `%debug` magic](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-debug)

---

## Mini-exemples supplémentaires

### Tracer la forme d'un objet

In [ ]:
data = {'nom': 'Alice', 'scores': [12, 15, 9]}
print(f'{type(data)=}')
print(f'{len(data)=}')
print(f'{list(data.keys())=}')

### `repr` vs `str`

In [ ]:
x = 'coucou'
print(str(x))   # pour l'utilisateur
print(repr(x))  # pour le debug, garde les guillemets

`repr` révèle les caractères invisibles — `\n`, `\t`, etc. Très utile quand un bug vient d'un espace insécable.

In [ ]:
mysterieux = 'hello\u00a0world'  # espace insécable
print(mysterieux)
print(repr(mysterieux))

### `%timeit` — mesurer un bout de code dans Jupyter

In [ ]:
# Dans un vrai notebook :
# %timeit sum(range(10000))
import timeit
t = timeit.timeit('sum(range(10000))', number=1000)
print(f'{t=:.3f}s')

### Impression de la pile

In [ ]:
import traceback

def a() -> None:
    b()

def b() -> None:
    traceback.print_stack()

a()

### Afficher toutes les variables locales

In [ ]:
def f(x: int, y: int) -> int:
    z = x + y
    print(f'{locals()=}')
    return z

f(3, 4)

### `icecream` — une alternative sympa

Le package tiers `icecream` fournit `ic(...)` qui fait `f'{var=}'` sur plusieurs expressions à la fois. Pratique pour du print-debug en série.

---

## Quiz flash — vérifiez vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre compréhension avant de passer au notebook suivant. Les réponses sont dans le bloc `<details>` en dessous.


**Question 1.** Que fait `breakpoint()` dans un script ?

<details>
<summary>📖 Réponse</summary>

Il invoque le débogueur intégré (`pdb` par défaut) — équivalent moderne à `import pdb; pdb.set_trace()`.

</details>

**Question 2.** Dans `pdb`, que signifie `n` ?

<details>
<summary>📖 Réponse</summary>

Exécuter l'instruction suivante, **sans** entrer dans les fonctions appelées.

</details>

**Question 3.** Et `s` ?

<details>
<summary>📖 Réponse</summary>

Exécuter l'instruction suivante, **en entrant** dans les fonctions appelées (step into).

</details>

**Question 4.** À quoi sert la syntaxe `f"{x=}"` ?

<details>
<summary>📖 Réponse</summary>

Elle affiche à la fois le nom et la valeur de `x`. Raccourci pour `f'x={x}'`.

</details>

**Question 5.** Quelle magie Jupyter lance un débugger post-mortem sur la dernière exception ?

<details>
<summary>📖 Réponse</summary>

`%debug` — à taper dans une cellule juste après l'erreur.

</details>

---

## Cheat sheet — Débogage express

| Technique | Quand l'utiliser |
|---|---|
| `print(f'{var=}')` | Inspection rapide d'une variable |
| `breakpoint()` | Inspection interactive au pas à pas |
| `%debug` (Jupyter) | Post-mortem après exception |
| `logger.exception(...)` | Tracer un bug en production |
| `traceback.print_stack()` | Voir la pile d'appels en cours |
| Git bisect | Trouver le commit qui a cassé |

### Mini-récap : les 3 questions avant de déboguer

1. **Quel est le plus petit code qui reproduit le bug ?**
2. **Quel est l'écart entre ce que j'attends et ce que j'observe ?**
3. **À quel moment exactement la valeur devient-elle fausse ?**

In [ ]:
# Exemple : tracer l'évolution d'une variable dans une boucle
somme = 0
for i in range(5):
    print(f'{i=} {somme=}')
    somme = somme + i
print(f'final {somme=}')